# 01 — The Agent Loop + Tools

**Production question:** *Can the agent act on real business data — not just chat?*

An LLM alone can only guess. An **agent** runs a loop: the model decides it needs data, calls a **tool**, reads the result, and repeats until it can answer.

```
Customer ─► LLM ─► tool call ─► tool result ─► LLM ─► … ─► answer
```

In Strands, a tool is just a Python function with `@tool`. The **docstring and type hints become the tool schema** the model reads, so there's no routing code to write.

### Step 1 — The backend tools

In real life these call your order-management system, payment gateway and ticketing system. Here they use mock data so the demo runs anywhere. **The agent doesn't know or care which one it's talking to.**

In [ ]:
from strands import tool

# --- Business constants (shared with the policy layer in steering.py) ---
RETURN_WINDOW_DAYS = 30
SUPERVISOR_REVIEW_THRESHOLD = 50_000  # ₹ — refunds at or above this need a human

# --- Mock data ---
CUSTOMERS = {
    "C-1001": {"name": "Priya Sharma", "city": "Bengaluru", "email": "priya.sharma@example.com",
               "phone": "+91 98450 12345", "account_status": "active"},
    "C-1002": {"name": "Arjun Reddy", "city": "Hyderabad", "email": "arjun.reddy@example.com",
               "phone": "+91 99490 67890", "account_status": "active"},
    "C-1003": {"name": "Kavya Iyer", "city": "Chennai", "email": "kavya.iyer@example.com",
               "phone": "+91 98410 24680", "account_status": "active"},
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Earbuds", "amount": 3499, "status": "Delivered",
         "order_date": "2026-10-14", "delivered_date": "2026-10-18", "tracking": "AWB-7781229034"},
        {"order_id": "ORD-5488", "item": "USB-C Hub (7-in-1)", "amount": 2199, "status": "Shipped",
         "order_date": "2026-10-26", "estimated_delivery": "2026-10-31", "tracking": "AWB-7781229871"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 6999, "status": "Delayed",
         "order_date": "2026-10-15", "estimated_delivery": "2026-10-21", "tracking": "AWB-7781227745"},
    ],
    "C-1003": [
        {"order_id": "ORD-5301", "item": "Smartwatch", "amount": 12999, "status": "Delivered",
         "order_date": "2026-09-02", "delivered_date": "2026-09-06", "tracking": "AWB-7781221108"},
        {"order_id": "ORD-5602", "item": "Laptop (14-inch, 16 GB RAM)", "amount": 72990, "status": "Delivered",
         "order_date": "2026-10-20", "delivered_date": "2026-10-23", "tracking": "AWB-7781230456"},
    ],
}


def find_order(order_id: str) -> dict | None:
    for orders in ORDERS.values():
        for order in orders:
            if order["order_id"] == order_id:
                return order
    return None


def inr(amount: float) -> str:
    return f"₹{amount:,.0f}"


# --- Tools ---

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer's profile by their customer ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"No customer found with ID {customer_id}"
    return (f"Customer: {c['name']} ({c['city']})\nEmail: {c['email']}\n"
            f"Phone: {c['phone']}\nAccount status: {c['account_status']}")


@tool
def get_order_history(customer_id: str) -> str:
    """Get the order history for a customer, including status, dates and tracking.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for o in orders:
        line = f"{o['order_id']}: {o['item']} — {inr(o['amount'])} [{o['status']}] ordered {o['order_date']}"
        if o.get("delivered_date"):
            line += f", delivered {o['delivered_date']}"
        if o.get("estimated_delivery"):
            line += f", estimated delivery {o['estimated_delivery']}"
        lines.append(line + f", tracking {o['tracking']}")
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order to the customer's original payment method.

    Args:
        order_id: The order ID to refund (e.g. ORD-5521)
        amount: The refund amount in Indian Rupees
    """
    return (f"Refund REF-{order_id[4:]} of {inr(amount)} initiated for {order_id} to the original "
            "payment method (UPI / card / net banking). It will reflect in 5-7 business days.")


@tool
def create_support_ticket(customer_id: str, reason: str, priority: str = "normal") -> str:
    """Escalate an issue to a human support specialist by creating a ticket.

    Args:
        customer_id: The customer ID (e.g. C-1001)
        reason: Short description of why this needs a human
        priority: "normal" or "high"
    """
    return (f"Ticket TKT-{abs(hash((customer_id, reason))) % 90000 + 10000} created for {customer_id} "
            f"({priority} priority). A specialist will respond within 24 hours.")


ALL_TOOLS = [lookup_customer, get_order_history, process_refund, create_support_ticket]

### Step 2 — The instructions

The system prompt gives the agent its role and ground rules. Notice that **today's date is injected at runtime**. Without it, the model can't tell whether an order is within the 30-day return window.

In [ ]:
import os
from datetime import date

# Swapping models is a config change: e.g. amazon.nova-pro-v1:0 or a Claude Haiku profile
MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")


def build_system_prompt() -> str:
    today = date.today().strftime("%d %B %Y (%A)")
    return f"""You are the customer support agent for ElectroBazaar, an online electronics store in India.
Today's date is {today}.

Be helpful, professional and concise. Use the available tools to look up customer
information and act on requests. All prices are in Indian Rupees (₹).

Guidelines:
- Always ask for the customer ID first if you don't have it.
- Verify the customer with lookup_customer before taking any action on their account.
- Use tool data to answer questions — never ask the customer for information you already have.
- If you have skills available, activate the one that matches the request and follow it.
- Remember preferences the customer shares (contact channel, refund method, language) and honour them.
- Be warm but efficient."""

### Step 3 — Create the agent and talk to it

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    tools=ALL_TOOLS,
    system_prompt=build_system_prompt(),
)

result = agent("Hi, I'm customer C-1001. Can you check on my recent orders?")

### Step 4 — Look inside the loop

The agent made several model calls and tool calls on its own. The conversation history shows exactly what happened, and the metrics show what it cost.

In [ ]:
import json

for i, msg in enumerate(agent.messages):
    for block in msg["content"]:
        if "text" in block and block["text"].strip():
            who = "👤 CUSTOMER " if msg["role"] == "user" else "🤖 AGENT    "
            print(f"[{i}] {who}: {block['text'][:90]!r}")
        elif "toolUse" in block:
            tu = block["toolUse"]
            print(f"[{i}] 🔧 TOOL CALL : {tu['name']}({json.dumps(tu['input'])})")
        elif "toolResult" in block:
            text = block["toolResult"]["content"][0].get("text", "")
            print(f"[{i}] 📦 RESULT    : {text[:90]!r}")

summary = result.metrics.get_summary()
print(f"\n📊 Loop cycles: {summary['total_cycles']}  |  Tokens: {result.metrics.accumulated_usage['totalTokens']}")

### Step 5 — A harder case

Arjun's keyboard was due on 21 October. The agent has to work out from the dates that the order is late, with nobody telling it.

In [ ]:
agent = Agent(model=BedrockModel(model_id=MODEL_ID), tools=ALL_TOOLS, system_prompt=build_system_prompt())
result = agent("I'm C-1002. My keyboard still hasn't arrived. What is going on?")

---
### What we have — and what's missing for production

✅ An agent that reasons over real data with **~10 lines of agent code**.

But ask yourself:
- What stops it from calling a tool **100 times** in a loop and running up the bill? → **02 Hooks**
- What stops it from refunding a **₹72,990 laptop** without a supervisor? → **03 Skills + Steering**
- What happens when the customer comes back **tomorrow**? → **04 Memory**

Next: **02_hooks.ipynb**